Tools-
1. Tavily
2. DuckDuckGo Search
3. PyPDFLoader
4. Requests-based web scraping

This notebook builds a LangGraph-based research analyst that plans, researches, scrapes, summarizes, and writes a final report with charts.

In [4]:
import json
import os
import re
import tempfile
from collections import Counter
from html.parser import HTMLParser
from pathlib import Path
from typing import TypedDict
from urllib.parse import urlparse

import langchain
import requests
from IPython.display import Markdown, display
from langchain_community.document_loaders import PyPDFLoader
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, StateGraph


if not hasattr(langchain, "debug"):
    langchain.debug = False


class _HTMLTextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts: list[str] = []

    def handle_data(self, data):
        text = data.strip()
        if text:
            self.parts.append(text)

    def get_text(self):
        return re.sub(r"\s+", " ", " ".join(self.parts)).strip()


class _DDGResultParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.results: list[dict] = []
        self._in_title = False
        self._current_href = ""
        self._current_text: list[str] = []

    def handle_starttag(self, tag, attrs):
        attrs_dict = dict(attrs)
        if tag == "a" and attrs_dict.get("class", "").startswith("result__a"):
            self._in_title = True
            self._current_href = attrs_dict.get("href", "")
            self._current_text = []

    def handle_data(self, data):
        if self._in_title:
            text = data.strip()
            if text:
                self._current_text.append(text)

    def handle_endtag(self, tag):
        if tag == "a" and self._in_title:
            title = re.sub(r"\s+", " ", " ".join(self._current_text)).strip()
            if self._current_href:
                self.results.append({"title": title, "url": self._current_href})
            self._in_title = False
            self._current_href = ""
            self._current_text = []


class ResearchState(TypedDict, total=False):
    topic: str
    plan: list[str]
    search_query: str
    tavily_results: list[dict]
    ddg_results: list[dict]
    urls: list[str]
    scraped_docs: list[dict]
    summary: str
    report: str
    chart_data: dict[str, int]


_tavily_api_key = os.getenv("TAVILY_API_KEY")
search_tavily = TavilySearch(max_results=5, tavily_api_key=_tavily_api_key) if _tavily_api_key else None


def _as_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)


def _extract_results(raw_value) -> list[dict]:
    if isinstance(raw_value, list):
        return [item if isinstance(item, dict) else {"content": _as_text(item)} for item in raw_value]
    if isinstance(raw_value, dict):
        for key in ("results", "items", "data"):
            if key in raw_value and isinstance(raw_value[key], list):
                return [item if isinstance(item, dict) else {"content": _as_text(item)} for item in raw_value[key]]
        return [raw_value]
    if isinstance(raw_value, str):
        try:
            parsed = json.loads(raw_value)
            return _extract_results(parsed)
        except json.JSONDecodeError:
            return [{"content": raw_value}]
    return [{"content": _as_text(raw_value)}]


def _extract_urls(results: list[dict]) -> list[str]:
    urls: list[str] = []
    for item in results:
        for key in ("url", "link", "source"):
            url = item.get(key)
            if isinstance(url, str) and url.startswith(("http://", "https://")):
                urls.append(url)
                break
    return list(dict.fromkeys(urls))


def _search_duckduckgo(query: str, max_results: int = 5) -> list[dict]:
    response = requests.get(
        "https://html.duckduckgo.com/html/",
        params={"q": query},
        timeout=20,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()
    parser = _DDGResultParser()
    parser.feed(response.text)
    results = parser.results[:max_results]
    return [{"title": item.get("title", ""), "url": item.get("url", "")} for item in results]


def _fetch_web_text(url: str, timeout: int = 20) -> str:
    response = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    extractor = _HTMLTextExtractor()
    extractor.feed(response.text)
    return extractor.get_text()


def _download_pdf(url: str) -> Path:
    response = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    suffix = ".pdf"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp_file:
        temp_file.write(response.content)
        return Path(temp_file.name)


def planner_node(state: ResearchState) -> ResearchState:
    topic = state["topic"].strip()
    return {
        "search_query": topic,
        "plan": [
            f"Clarify the research scope for {topic}",
            "Search the web with Tavily and DuckDuckGo",
            "Scrape the strongest pages and any PDFs",
            "Summarize the evidence into findings",
            "Write a final report with charts",
        ],
    }


def research_node(state: ResearchState) -> ResearchState:
    query = state["search_query"]
    tavily_results: list[dict] = []
    if search_tavily is not None:
        try:
            tavily_results = _extract_results(search_tavily.invoke(query))
        except Exception as exc:
            tavily_results = [{"content": f"Tavily search failed: {exc}"}]
    ddg_results = _extract_results(_search_duckduckgo(query))
    urls = _extract_urls(tavily_results + ddg_results)
    return {
        "tavily_results": tavily_results,
        "ddg_results": ddg_results,
        "urls": urls,
    }


def scraper_node(state: ResearchState) -> ResearchState:
    scraped_docs: list[dict] = []
    for url in state.get("urls", []):
        try:
            if url.lower().endswith(".pdf"):
                pdf_path = _download_pdf(url)
                loader = PyPDFLoader(str(pdf_path))
                pages = loader.load()
                text = "\n\n".join(page.page_content for page in pages)
                scraped_docs.append({"url": url, "type": "pdf", "text": text[:12000]})
                pdf_path.unlink(missing_ok=True)
            else:
                text = _fetch_web_text(url)
                scraped_docs.append({"url": url, "type": "html", "text": text[:12000]})
        except Exception as exc:
            scraped_docs.append({"url": url, "type": "error", "text": str(exc)})
    return {"scraped_docs": scraped_docs}


def summarizer_node(state: ResearchState) -> ResearchState:
    docs = state.get("scraped_docs", [])
    useful_docs = [doc for doc in docs if doc.get("type") in {"html", "pdf"} and doc.get("text")]
    top_notes: list[str] = []
    for doc in useful_docs[:5]:
        snippet = re.sub(r"\s+", " ", doc["text"][:400]).strip()
        top_notes.append(f"- {doc['url']}: {snippet}")

    evidence_block = top_notes if top_notes else ["- No readable source text was collected."]
    summary = "\n".join(
        [
            f"## Research Summary for {state['topic']}",
            "",
            "### Plan",
            *[f"- {step}" for step in state.get("plan", [])],
            "",
            "### Key Evidence",
            *evidence_block,
        ]
    )
    return {"summary": summary}


def writer_node(state: ResearchState) -> ResearchState:
    docs = state.get("scraped_docs", [])
    source_types = Counter(doc.get("type", "unknown") for doc in docs)
    chart_data = dict(source_types)

    report_lines = [
        f"# AI Research Analyst Report: {state['topic']}",
        "",
        "## Workflow",
        *[f"- {step}" for step in state.get("plan", [])],
        "",
        "## Summary",
        state.get("summary", ""),
        "",
        "## Source Mix",
        *[f"- {source_type}: {count}" for source_type, count in chart_data.items()],
    ]
    return {"report": "\n".join(report_lines), "chart_data": chart_data}


workflow = StateGraph(ResearchState)
workflow.add_node("planner", planner_node)
workflow.add_node("research", research_node)
workflow.add_node("scraper", scraper_node)
workflow.add_node("summarizer", summarizer_node)
workflow.add_node("writer", writer_node)
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "research")
workflow.add_edge("research", "scraper")
workflow.add_edge("scraper", "summarizer")
workflow.add_edge("summarizer", "writer")
workflow.add_edge("writer", END)
app = workflow.compile()

In [5]:
topic = "AI research analyst workflow for web research and report generation"
result = app.invoke({"topic": topic})

print(result["report"])
display(Markdown(result["summary"]))

chart_items = list(result.get("chart_data", {}).items())
if chart_items:
    markdown_table = ["| source_type | count |", "| --- | ---: |"]
    markdown_table.extend(f"| {source_type} | {count} |" for source_type, count in chart_items)
    display(Markdown("\n".join(markdown_table)))
else:
    display(Markdown("No source data was collected."))

# AI Research Analyst Report: AI research analyst workflow for web research and report generation

## Workflow
- Clarify the research scope for AI research analyst workflow for web research and report generation
- Search the web with Tavily and DuckDuckGo
- Scrape the strongest pages and any PDFs
- Summarize the evidence into findings
- Write a final report with charts

## Summary
## Research Summary for AI research analyst workflow for web research and report generation

### Plan
- Clarify the research scope for AI research analyst workflow for web research and report generation
- Search the web with Tavily and DuckDuckGo
- Scrape the strongest pages and any PDFs
- Summarize the evidence into findings
- Write a final report with charts

### Key Evidence
- https://docs.gumloop.com/nodes/using_ai/ai_web_research: AI Web Research - Gumloop !function(){var b=""; function a(p){if(p==null)return"/";p=""+p;if(""===p)return"/";return"/"===p.charAt(0)?p:"/"+p} function u(p){if(p==null)return p

## Research Summary for AI research analyst workflow for web research and report generation

### Plan
- Clarify the research scope for AI research analyst workflow for web research and report generation
- Search the web with Tavily and DuckDuckGo
- Scrape the strongest pages and any PDFs
- Summarize the evidence into findings
- Write a final report with charts

### Key Evidence
- https://docs.gumloop.com/nodes/using_ai/ai_web_research: AI Web Research - Gumloop !function(){var b=""; function a(p){if(p==null)return"/";p=""+p;if(""===p)return"/";return"/"===p.charAt(0)?p:"/"+p} function u(p){if(p==null)return p;p=""+p;if(!p||p.charAt(p.length-1)==="/")return p.slice(0,-1);return p} function i(p){if(p==null)return p;p=""+p;if(6<=p.length&&p.substring(p.length-6)==="/index")return p.substring(0,p.length-6);if("index"===p)return"";re
- https://www.mindstudio.ai/blog/ai-agents-research-analysis: 9 AI Agents for Research and Analysis | MindStudio (function(){const GTM_ID = "GTM-MGRB9WXW"; (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start': new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0], j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src= 'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f); })(window,document,'
- https://paperguide.ai: Paperguide: The AI Research Assistant document.querySelectorAll('body link[rel="icon"], body link[rel="apple-touch-icon"]').forEach(el => document.head.appendChild(el)) (self.__next_s=self.__next_s||[]).push([0,{"type":"application/ld+json","children":"{\"@context\":\"http://schema.org\",\"@type\":\"WebSite\",\"name\":\"Paperguide\",\"url\":\"https://paperguide.ai/\"}","id":"website-json-ld"}]) (s
- https://doreveal.com: DoReveal - Transforming Research with AI-Powered Insights { "imports": { "application": "/assets/application-70737591afb1d71370ec2e1ecea4e32c5fa746e4e14baf6e6a91e07a908e38a6.js", "@rails/ujs": "/assets/@rails--ujs-8fbd650218b6b146a78aa5d7d6cd8eb8d9da87cc6d504aab7fc3c6b64e94d34b.js", "@hotwired/turbo-rails": "/assets/@hotwired--turbo-rails-3d2bf275df32fbe55dc4f711d13a7a99737900f08d493ed5ca39a8749f4

| source_type | count |
| --- | ---: |
| html | 4 |
| error | 1 |